# M10 · Graph-Augmented RAG evaluation on Colab

Notebook này chạy ablation Hybrid vs Hybrid + Graph trên pilot set, hiển thị chunk/provenance để kiểm tra, và có một cell tùy chọn gọi Qwen để xem câu trả lời cùng citation.

Project phải được đồng bộ lên Google Drive tại `ML_Project/vn-landlaw-qa` (có thể sửa `PROJECT_ROOT`). Dùng Runtime GPU nếu muốn chạy Qwen; retrieval evaluation cũng chạy được trên CPU.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, json

drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/ML_Project/vn-landlaw-qa')  # sửa nếu cần
assert (PROJECT_ROOT / 'scripts/evaluate_graph.py').is_file(), f'Không tìm thấy project: {PROJECT_ROOT}'
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('Project:', PROJECT_ROOT)

In [ ]:
%pip install -q -r requirements/retrieval.txt

# Nếu vừa cài packages và Colab yêu cầu restart runtime, restart rồi chạy lại cell mount Drive.

In [ ]:
import torch
required = [
    'data/processed/chunks.json',
    'data/evaluation/graph_retrieval.jsonl',
    'indexes/bm25/bm25_index.json.gz',
    'indexes/faiss/faiss.index',
]
missing = [p for p in required if not (PROJECT_ROOT / p).is_file()]
assert not missing, f'Thiếu artifacts: {missing}; đồng bộ dữ liệu/index lên Drive trước.'
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
print('Pilot questions:', sum(1 for _ in open(PROJECT_ROOT / 'data/evaluation/graph_retrieval.jsonl', encoding='utf-8')))

## 1. Retrieval ablation

Cell này tải multilingual-e5-base nếu cache Colab chưa có, dùng index hiện tại và ghi báo cáo JSON vào `artifacts/evaluation/graph_ablation_colab.json`. Lần đầu có thể mất thời gian tải model.

In [ ]:
import subprocess
device = 'cuda' if torch.cuda.is_available() else 'cpu'
cmd = [sys.executable, 'scripts/evaluate_graph.py', '--device', device, '--top-k', '8',
       '--output', 'artifacts/evaluation/graph_ablation_colab.json']
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)
report = json.loads((PROJECT_ROOT / 'artifacts/evaluation/graph_ablation_colab.json').read_text(encoding='utf-8'))
print('Metrics:', json.dumps(report['systems'], ensure_ascii=False, indent=2))
print('By relation:', json.dumps(report['by_relation_focus'], ensure_ascii=False, indent=2))

In [ ]:
import pandas as pd
from IPython.display import display
chunks = json.loads((PROJECT_ROOT / 'data/processed/chunks.json').read_text(encoding='utf-8'))
chunk_by_id = {str(item['chunk_id']): item for item in chunks}
rows = []
for case in report['cases']:
    annotations = {item['chunk_id']: item for item in case.get('graph_hit_annotations', [])}
    for rank, chunk_id in enumerate(case['hybrid_graph_chunk_ids'], start=1):
        chunk = chunk_by_id[chunk_id]
        annotation = annotations.get(chunk_id, {})
        meta = chunk.get('metadata', {})
        rows.append({
            'question_id': case['question_id'], 'rank': rank, 'chunk_id': chunk_id,
            'document': meta.get('document_id'), 'article': meta.get('article'),
            'clause': meta.get('clause'), 'relations': ', '.join(annotation.get('relations', [])),
            'hops': annotation.get('hops', 0), 'provenance': annotation.get('provenance', []),
            'text': chunk.get('text', '')[:350].replace('\n', ' '),
        })
display(pd.DataFrame(rows))
print('Relation provenance in report:')
for case in report['cases']:
    if case['graph_provenance']:
        print(case['question_id'], case['graph_provenance'][:3])

## 2. Optional end-to-end QA output (T4 recommended)

Cell dưới nạp Qwen 4-bit như M9. Nó không cần STT/TTS; gọi trực tiếp RAG để xem answer, validated citations và graph-added sources. Model download lần đầu có thể lớn.

In [ ]:
%pip install -q --upgrade git+https://github.com/huggingface/transformers.git@main accelerate bitsandbytes
# Không cài lại torch; giữ bản CUDA có sẵn của Colab.

In [ ]:
from landlaw_rag.audio.runtime import load_colab_rag

assert torch.cuda.is_available(), 'QA demo cần Runtime GPU (T4). Có thể bỏ qua cell này nếu chỉ đánh giá retrieval.'
rag = load_colab_rag(PROJECT_ROOT, model_name='Qwen/Qwen3.5-4B', strict_citations=True)
sample_questions = [
    'Luật 43/2024 sửa đổi thời điểm có hiệu lực của Luật Đất đai 2024 như thế nào?',
    'Theo Nghị quyết số 254/2025/QH15, hồ sơ xác định giá đất cụ thể chưa có quyết định thì xử lý thế nào?',
]
qa_results = []
for question in sample_questions:
    result = rag(question)
    qa_results.append(result)
    print('\nQUESTION:', question)
    print('ANSWER:', result['answer'])
    print('CITATIONS:', result['citations'])
    print('SOURCES:', [{
        'chunk_id': source.get('chunk_id'),
        'document_id': source.get('metadata', {}).get('document_id'),
        'article': source.get('metadata', {}).get('article'),
        'clause': source.get('metadata', {}).get('clause'),
        'graph_relations': source.get('graph_relations', []),
        'graph_provenance': source.get('graph_provenance', []),
    } for source in result['sources']])

## Chấm citation và answer

Đọc answer và source mapping thực tế. Điền `answer_text`, `cited_chunk_ids`, `answer_correct` và `citation_supported` trong `data/evaluation/answer_judgments.template.jsonl`, rồi chạy lại evaluator với `--judgments data/evaluation/answer_judgments.jsonl`. Citation source precision/recall là mức trùng với legal-unit labels; claim-level support vẫn cần người chấm đọc căn cứ.